# Imports

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.linear_model import LinearRegression
from tqdm import tqdm
from sklearn.metrics import make_scorer, matthews_corrcoef, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from lightgbm import LGBMRegressor
from lazypredict.Supervised import LazyClassifier
from transformers import AutoModelForSequenceClassification, AutoModel
from huggingface_hub import notebook_login
from typing import Tuple

# Dataset Initialization and Creation

In [ ]:
# Load your chosen dataset from HuggingFace
dataset_name = "GleghornLab/fitness_prediction"
data = load_dataset(dataset_name)
data

In [19]:
df_train = pd.DataFrame(data["train"]).drop_duplicates(subset=['seqs'])
df_test = pd.DataFrame(data["test"]).drop_duplicates(subset=['seqs'])
df_valid = pd.DataFrame(data["valid"]).drop_duplicates(subset=['seqs'])
seq_list_train = df_train['seqs'].tolist()
seq_list_test = df_test['seqs'].tolist()
seq_list_valid = df_valid['seqs'].tolist()

# We extract our labels and sequences from our target dataset
# Also note if you simply want to test a preexisting models embeddings that have already been transformed, you can run the first cell of model training and then skip to the hyperparameter tuning section

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "Synthyra/ESM2-650M"
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch.float16).cuda()

In [ ]:
all_seqs = data['train']['seqs'] + data['valid']['seqs'] + data['test']['seqs']
print(f"Number of sequences: {len(all_seqs)}")
emb_dict = model.embed_dataset(
  sequences=all_seqs, # list of protein strings
  batch_size=8, # embedding batch size
  max_len=2048, # truncate to max_len
  full_embeddings=True, # return residue-wise embeddings
  full_precision=False, # store as float32
  num_workers=10, # data loading num workers
  sql=False, # return dictionary of sequences and embeddings
)

In [24]:
class Pooler:
    def __init__(self, hf_dataset, emb_dict):
        self.train_seqs, self.train_labels = hf_dataset['train']['seqs'], hf_dataset['train']['labels']
        self.valid_seqs, self.valid_labels = hf_dataset['valid']['seqs'], hf_dataset['valid']['labels']
        self.test_seqs, self.test_labels = hf_dataset['test']['seqs'], hf_dataset['test']['labels']
        self.emb_dict = emb_dict

        self.pooling_options = {
            'mean': self.mean_pooling,
            'max': self.max_pooling,
            'min': self.min_pooling,
            'norm': self.norm_pooling,
            'prod': self.prod_pooling,
            'median': self.median_pooling,
            'std': self.std_pooling,
            'var': self.var_pooling,
            'cls': self.cls_pooling,
            'eos': self.eos_pooling
        }

    def mean_pooling(self, emb): # (L, d) -> (d)
        return emb.mean(dim=0)

    def max_pooling(self, emb): # (L, d) -> (d)
        return emb.max(dim=0).values

    def min_pooling(self, emb): # (L, d) -> (d)
        return emb.min(dim=0).values

    def norm_pooling(self, emb): # (L, d) -> (d)
        return emb.norm(dim=0, p=2)

    def prod_pooling(self, emb): # (L, d) -> (d)
        length = emb.shape[0]
        return emb.prod(dim=0) / length

    def median_pooling(self, emb): # (L, d) -> (d)
        return emb.median(dim=0).values

    def std_pooling(self, emb): # (L, d) -> (d)
        return emb.std(dim=0)

    def var_pooling(self, emb): # (L, d) -> (d)
        return emb.var(dim=0)

    def cls_pooling(self, emb): # (L, d) -> (d)
        return emb[0, :]

    def eos_pooling(self, emb): # (L, d) -> (d)
        return emb[-1, :]

    def get_pooled_embeddings(self, pooling_types: Tuple[str, str]): # [mean, max]
        """
        pooler = Pooler(hf_dataset, emb_dict)
        pooling_types = ['mean', 'max']
        X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)
        # X (n_samples, n_features)
        # y (n_samples)
        """
        for pool_type in pooling_types:
            if pool_type not in self.pooling_options:
                raise ValueError(f"Invalid pooling type: {pool_type}")

        train_embeddings, valid_embeddings, test_embeddings = [], [], []
        train_labels = np.array(self.train_labels)
        valid_labels = np.array(self.valid_labels)
        test_labels = np.array(self.test_labels)

        # concatenate all pooling for each sequence
        for seq in self.train_seqs:
            final_emb = []
            emb = self.emb_dict[seq] # (L, d)
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb) # (d)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten() # (d * len(pooling_type))
            train_embeddings.append(final_emb)

        train_embeddings = torch.stack(train_embeddings).numpy().astype(np.float32)

        for seq in self.valid_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            valid_embeddings.append(final_emb)

        valid_embeddings = torch.stack(valid_embeddings).numpy().astype(np.float32)

        for seq in self.test_seqs:
            final_emb = []
            emb = self.emb_dict[seq]
            for pool_type in pooling_types:
                pooled_emb = self.pooling_options[pool_type](emb)
                final_emb.append(pooled_emb)
            final_emb = torch.cat(final_emb, dim=0).flatten()
            test_embeddings.append(final_emb)

        test_embeddings = torch.stack(test_embeddings).numpy().astype(np.float32)

        return train_embeddings, train_labels, valid_embeddings, valid_labels, test_embeddings, test_labels


In [25]:
pooling_types = ['mean']
pooler = Pooler(data, emb_dict)
X_train, y_train, X_valid, y_valid, X_test, y_test = pooler.get_pooled_embeddings(pooling_types)

In [31]:
def mcc_metric(y_true, y_pred):
    return matthews_corrcoef(y_true, y_pred)

# Set a display name for the function
mcc_metric.__name__ = "mcc_metric"

# Create a sklearn scorer for MCC
mcc_scorer = make_scorer(mcc_metric, greater_is_better=True)

def r2_metric(y_true, y_pred):
    return r2_score(y_true, y_pred)

r2_metric.__name__ = "r2_metric"
r2_scorer = make_scorer(r2_metric, greater_is_better=True)

In [15]:
model = RandomForestClassifier() # You can use any other model provided but it must be imported

param_grid = {
    'n_jobs': [-1],
    'n_estimators': [100, 200, 300, 500, 750, 1000],  # Number of trees in the forest
    'max_depth': [5, 10, 15, 20, 25, 30, 40, 50, 75, 100],  # Maximum depth of the tree
    'min_samples_split': [2, 3, 5, 7, 10, 15, 20],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 3, 4, 5, 7, 10],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2', None, 0.5, 0.7, 0.9],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False],  # Whether bootstrap samples are used when building trees
    'criterion': ['gini', 'entropy', 'log_loss'],  # Function to measure the quality of a split
    'class_weight': ['balanced', 'balanced_subsample', None],  # Weights associated with classes
    'warm_start': [True, False],  # Whether to reuse previous solution
    'oob_score': [True, False],  # Whether to use out-of-bag samples
    'ccp_alpha': [0.0, 0.1, 0.2, 0.3],  # Complexity parameter for pruning
}
# Provide a parameter grid to test a wide range of parameters for your chosen model to optimize

grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=5, scoring=mcc_scorer, verbose=4, n_iter = 64) # Creates a grid search instance, note that the scoring method can also be adjusted here

In [ ]:
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_
best_score = grid_search.best_score_

print(best_params)
print(best_estimator) # Shows the best hyperparameters for the model

In [38]:
model = RandomForestRegressor() # You can use any other model provided but it must be imported

# Define the parameter grid
param_grid = {
    'n_jobs': [-1],
    'n_estimators': [100, 200, 300, 500, 750, 1000],  # Number of trees in the forest
    'max_depth': [5, 10, 20, 30, 50, 75, 100, None],  # Maximum depth of the tree
    'min_samples_split': [2, 5, 10, 15, 20, 25],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4, 6, 8, 10],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2', None, 0.5, 0.7, 0.9],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False],  # Whether bootstrap samples are used when building trees
    'criterion': ['squared_error', 'friedman_mse', 'poisson', 'absolute_error'],  # Split quality metrics
    'warm_start': [True, False],  # Whether to reuse previous solution
    'oob_score': [True, False],  # Whether to use out-of-bag samples
    'ccp_alpha': [0.0, 0.1, 0.2, 0.3],  # Complexity parameter for pruning
}
# Provide a parameter grid to test a wide range of parameters for your chosen model to optimize

grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, cv=5, scoring=r2_scorer, verbose=4, n_iter = 64) # Creates a grid search instance, note that the scoring method can also be adjusted here

In [ ]:
grid_search.fit(X_train, y_train) # Fits the parameter grid on a random set of hyperparameters provided in the parameter grid

In [ ]:
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_
best_score = grid_search.best_score_

print(best_params)
print(best_estimator) # Shows the best hyperparameters for the model